### Attention Layer and Deep Classifier


In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

# 1. Load the Fused Data
fusion_df = pd.read_csv("../Outputs/fusion_dataset.csv")

# 2. Prepare Tensors
print("Preparing dataset for PyTorch...")
# Extract features (everything except image_name and label)
feature_cols = [col for col in fusion_df.columns if col not in ['image_name', 'label']]
X = fusion_df[feature_cols].values.astype(np.float32)
y = fusion_df['label'].values.astype(np.float32).reshape(-1, 1)
image_names = fusion_df['image_name'].values

# Standardize features (Neural networks fail if features are on different scales)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_tensor = torch.tensor(X_scaled)
y_tensor = torch.tensor(y)

# Create a DataLoader to feed data to the network in batches
dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# 3. Define the PyTorch Model Architecture
class HybridAttentionClassifier(nn.Module):
    def __init__(self, input_dim):
        super(HybridAttentionClassifier, self).__init__()
        
        # Lightweight Attention Layer
        self.attention = nn.Sequential(
            nn.Linear(input_dim, input_dim // 2),
            nn.Tanh(),
            nn.Linear(input_dim // 2, input_dim),
            nn.Softmax(dim=1) # Converts outputs to probability-like weights
        )
        
        # Classification Head (Dense -> Dropout -> Dense -> Sigmoid)
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid() # Squeezes final output between 0 (Benign) and 1 (Malignant)
        )

    def forward(self, x):
        # 1. Calculate attention weights
        attn_weights = self.attention(x)
        # 2. Multiply original features by their learned importance
        fused_features = x * attn_weights
        # 3. Classify the weighted features
        out = self.classifier(fused_features)
        return out

input_dim = X.shape[1]
model = HybridAttentionClassifier(input_dim)

# 4. Training Setup (Using parameters from your methodology plan)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
epochs = 20

print(f"\nTraining Hybrid Attention Classifier on {input_dim} features...")

# 5. Training Loop
model.train()
for epoch in range(epochs):
    epoch_loss = 0.0
    for batch_X, batch_y in dataloader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    # Print update every 5 epochs to track progress
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss/len(dataloader):.4f}")

print("\nTraining Complete!")

# 6. Generate Final Predictions
model.eval()
with torch.no_grad():
    probabilities = model(X_tensor).numpy().flatten()
    # Threshold at 0.5: > 50% means Malignant (1), else Benign (0)
    predictions = (probabilities > 0.5).astype(int)

# 7. Package and Export Results
results_df = pd.DataFrame({
    'image_name': image_names,
    'true_label': y.flatten().astype(int),
    'predicted_label': predictions,
    'probability': probabilities
})

output_preds_path = "../Outputs/predictions.csv"
results_df.to_csv(output_preds_path, index=False)

print(f"SUCCESS! Final evaluation file saved to: {output_preds_path}")

Preparing dataset for PyTorch...

Training Hybrid Attention Classifier on 1298 features...
Epoch 1/20 | Loss: 0.6392
Epoch 5/20 | Loss: 0.1096
Epoch 10/20 | Loss: 0.0363
Epoch 15/20 | Loss: 0.0152
Epoch 20/20 | Loss: 0.0091

Training Complete!
SUCCESS! Final evaluation file saved to: ../Outputs/predictions.csv
